# v2 기업관계·뉴스 벡터 통합 및 CSV 대분류

기존 v3 Section 노드가 v2에 모두 있는지 검사한 뒤 `id.startswith("industry:")`인 노드만 제거합니다. 뉴스 768차원 임베딩은 기존 캐시에서 읽어 JSONL로 추출하고 v2 노드 마지막에 붙입니다. Industry를 가리키는 트리플을 정리하고 News → RELATED_TO → ParentCompany를 추가합니다.

CSV 대분류는 **기업관계의 Company → Section 연결**을 우선 사용합니다. 연결이 없는 기업은 노드의 업종·사업내용에 아래 규칙을 적용하고, 노드가 없으면 `모름`으로 기록합니다. 복수 대분류는 세미콜론으로 구분합니다. 기존 CSV 값·행은 보존합니다.


In [ ]:
SECTION_ROWS = [
    {"name": "금융", "category": ["SPC", "펀드", "지주", "은행", "증권", "보험", "금융", "투자", "유동화", "신탁", "대출", "자산운용", "캐피탈", "대부", "여신", "할부", "집합투자", "사모", "특수목적", "기업어음", "벤처", "조합", "손해사정"]},
    {"name": "제조", "category": ["전자", "자동차", "화학", "소재", "기계", "제조", "반도체", "전지", "케이블", "시멘트", "레미콘", "플라스틱", "철강", "제강", "금속", "섬유", "의류", "선박", "부품"]},
    {"name": "IT·미디어", "category": ["소프트웨어", "통신", "게임", "방송", "콘텐츠", "IT서비스", "정보통신", "정보기술", "컴퓨터", "데이터베이스", "시스템", "광고", "영화", "드라마", "음악", "엔터테인먼트", "매니지먼트", "연예", "미디어", "컨텐츠", "출판", "인터넷", "플랫폼"]},
    {"name": "부동산·건설", "category": ["부동산", "임대", "건설", "시공", "개발사업", "시행", "분양", "주택", "건축", "엔지니어링"]},
    {"name": "서비스", "category": ["호텔", "교육", "컨설팅", "연구", "정비", "서비스", "골프", "여행", "학원", "시설관리", "콜센터", "텔레마케팅", "고객센터", "사업지원", "경영", "경비", "청소", "인력", "전시회", "전시대행", "테마파크", "콘도"]},
    {"name": "유통·물류", "category": ["도소매", "무역", "운송", "창고", "유통", "물류", "도매", "소매", "도ㆍ소매", "운수", "항만", "해운", "택배", "백화점"]},
    {"name": "바이오·헬스케어", "category": ["제약", "의료", "화장품", "의약품", "바이오", "헬스", "병원", "건강"]},
    {"name": "에너지·환경", "category": ["발전", "태양광", "폐기물", "가스", "에너지", "전력", "전기업", "석유", "원유", "연료", "열공급", "증기", "환경"]},
    {"name": "식품·농업", "category": ["식품", "외식", "농축산", "음료", "음식", "구내식당", "급식", "정육", "농업", "축산", "양돈", "가금류", "작물", "수산", "어업", "생수", "주류", "사료"]},
    {"name": "모름", "category": []},
]


def classify_industry(industry_name):
    normalized_name = (industry_name or "").casefold()
    matches = [
        section["name"]
        for section in SECTION_ROWS
        if section["name"] != "모름"
        and any(keyword.casefold() in normalized_name for keyword in section["category"])
    ]
    return matches or ["모름"]

In [ ]:
import json
from pathlib import Path


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp")
    try:
        with temporary_path.open("w", encoding="utf-8", newline="\n") as file:
            for row in rows:
                file.write(json.dumps(row, ensure_ascii=False) + "\n")
        temporary_path.replace(path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def build_section_nodes(section_rows=SECTION_ROWS):
    return [
        {
            "id": f"section:{section['name']}",
            "type": "Section",
            "properties": {
                "name": section["name"],
                "category": list(section["category"]),
            },
        }
        for section in section_rows
    ]


def build_section_triples(assignments, section_rows=SECTION_ROWS):
    section_order = {row["name"]: index for index, row in enumerate(section_rows)}
    section_triples = []
    for assignment in sorted(
        assignments, key=lambda row: (row["subject_type"], row["subject"])
    ):
        subject = assignment.get("subject")
        subject_type = assignment.get("subject_type")
        if not subject:
            raise ValueError("Neo4j Company 노드에 id가 없습니다.")
        if subject_type not in {"ParentCompany", "SubsidiaryCompany"}:
            raise ValueError(f"허용되지 않은 회사 타입: {subject_type}")
        industry_names = sorted(
            {name for name in assignment.get("industry_names", []) if name}
        )
        matched_sections = set()
        for industry_name in industry_names:
            matched_sections.update(classify_industry(industry_name))
        matched_sections.discard("모름")
        if not matched_sections:
            matched_sections = {"모름"}

        industry_text = ", ".join(industry_names)
        for section_name in sorted(matched_sections, key=section_order.get):
            section_triples.append(
                {
                    "subject": subject,
                    "subject_type": subject_type,
                    "relation": "IN_INDUSTRY",
                    "object": f"section:{section_name}",
                    "object_type": "Section",
                    "source_case": "section_insert_v3",
                    "source_row": assignment.get("source_row"),
                    "evidence": industry_text,
                }
            )
    return section_triples


def validate_section_overlay(section_nodes, section_triples):
    section_ids = {node["id"] for node in section_nodes}
    if len(section_ids) != len(section_nodes):
        raise ValueError("중복된 노드 id가 있습니다.")
    seen = set()
    for triple in section_triples:
        object_id = triple.get("object")
        if object_id not in section_ids:
            raise ValueError(f"Section object를 찾을 수 없습니다: {object_id}")
        if triple.get("object_type") != "Section":
            raise ValueError(f"object_type 불일치: {object_id}")
        if triple.get("subject_type") not in {"ParentCompany", "SubsidiaryCompany"}:
            raise ValueError(f"subject_type 불일치: {triple.get('subject')}")
        key = (triple.get("subject"), triple.get("relation"), object_id)
        if key in seen:
            raise ValueError(f"중복된 트리플: {key}")
        seen.add(key)
    return {
        "section_node_count": len(section_nodes),
        "section_triple_count": len(section_triples),
    }

## 로컬 파일 준비

최종_기업개요.csv, 최종_종속기업_정리.csv, 최종_모기업_계열사_종속기업_통합.csv에 `대분류`를 추가합니다. 통합 CSV에는 `top_대분류`, `affiliate_대분류`, `subsidiary_대분류`도 추가하며 `대분류`는 이 세 역할의 분류 합집합입니다. `최종_*.csv`는 더 정리된 버전이며 속성 보강 출처로도 쓰입니다. 종속기업 CSV의 crno는 모기업 번호이므로 종속기업 식별자로 사용하지 않습니다.

원본 백업은 data/work/graph_v2_backups 아래에 생성합니다. 아래 셀은 DB에 연결하지 않습니다.


In [ ]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (start, *start.parents) if (p / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.neo4j.prepare_graph_v2 import prepare_files

preparation_report = prepare_files(PROJECT_ROOT)
display(preparation_report)


In [ ]:
import pandas as pd

for filename in ["최종_기업개요.csv", "최종_종속기업_정리.csv", "최종_모기업_계열사_종속기업_통합.csv"]:
    frame = pd.read_csv(PROJECT_ROOT / "data" / "clean" / filename, dtype=str, keep_default_na=False)
    columns = [c for c in frame if c in {"crno", "corpNm", "sbrdEnpNm", "top_corpNm", "affiliate_corpNm", "subsidiary_name"} or "대분류" in c]
    print(filename, len(frame), "행")
    display(frame[columns].head(10))


## 기존 load_graph.py로 Aura 적재

준비된 v2 노드·트리플을 적재합니다. 노드는 MERGE하고 속성은 SET으로 반영합니다. 대표자·별칭 등 추가 속성은 노드 JSONL에도 저장되어 있습니다. 뉴스 벡터 인덱스 news_vec도 생성합니다. 아래 스위치는 노트북 재실행 시 DB 쓰기를 구분하기 위한 설정입니다.


In [ ]:
from src.neo4j.load_graph import import_graph

APPLY_CHANGES = False
if APPLY_CHANGES:
    display(import_graph(target="aura"))
else:
    print("파일 준비 완료. Aura 적재 시 APPLY_CHANGES = True 또는 python src/neo4j/load_graph.py --target aura 실행")
